# EEG Pain-Level Classification with LightGBM

A full pipeline for classifying subjective pain levels from EEG data using Discrete Wavelet Transform (DWT) feature extraction and a LightGBM classifier.

**Pipeline overview:**
1. [Environment Setup](#1-environment-setup)
2. [EEG Data Loading & Inspection](#2-eeg-data-loading--inspection)
3. [Preprocessing & Artifact Removal (ICA)](#3-preprocessing--artifact-removal-ica)
4. [Epoching & Visualisation](#4-epoching--visualisation)
5. [DWT Feature Extraction](#5-dwt-feature-extraction)
6. [Merging Per-Participant CSV Files](#6-merging-per-participant-csv-files)
7. [Dataset Preparation](#7-dataset-preparation)
8. [LightGBM Model Training & Evaluation](#8-lightgbm-model-training--evaluation)
9. [Model Saving](#9-model-saving)


## 1. Environment Setup

In [ ]:
# Install required packages
# MNE: EEG/MEG analysis library
# autoreject: automatic rejection threshold estimation for EEG epochs
# PyWavelets: Discrete Wavelet Transform implementation
# lightgbm: gradient boosting classifier
%pip install mne autoreject PyWavelets lightgbm


In [ ]:
# Mount Google Drive to access EEG data files and save outputs
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne

# Suppress verbose MNE logs — set to 'warning' or 'info' for more detail
mne.set_log_level('error')

## 2. EEG Data Loading & Inspection

Load a single BrainVision `.vhdr` file and inspect its annotations.
Each annotation encodes a pain stimulus level (e.g. `Comment/10` = pain level 10).


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# Update p_id to match the participant you want to process.
p_id      = 'vp51'
data_dir  = '/content/drive/My Drive/EEG_Data/'
raw_file  = os.path.join(data_dir, f'Exp_Mediation_Paradigm1_Perception_{p_id}.vhdr')

# Load raw EEG recording; preload=True reads all data into memory up front
raw = mne.io.read_raw_brainvision(raw_file, preload=True, verbose='error')
print(raw.info)

In [ ]:
# Inspect annotations to understand which pain levels are present in this file.
# Run this before defining event_mapping — the available Comment/* codes vary
# between participants.
mne.events_from_annotations(raw)

In [ ]:
events, event_dict = mne.events_from_annotations(raw)

# Map annotation strings to integer event IDs.
# Adjust this dictionary based on the output of the cell above.
event_mapping = {
    'Comment/10': 10001,
    'Comment/20': 10002,
    'Comment/30': 10003,
    'Comment/40': 10004,
    'Comment/50': 10006,
    'Comment/60': 10007,
}

# Plot the event timeline to verify the mapping visually
fig, ax = plt.subplots(figsize=(15, 5))
mne.viz.plot_events(events, raw.info['sfreq'], event_id=event_mapping, axes=ax)
plt.title(f'Event Timeline — Participant {p_id}')
plt.show()

## 3. Preprocessing & Artifact Removal (ICA)

Steps performed here:
1. Assign correct channel types (EOG, ECG, misc) so MNE can apply them correctly.
2. Set the standard 10-05 electrode montage.
3. Band-pass filter the raw signal (0.1 – 30 Hz for general analysis; 1 – 30 Hz for ICA).
4. Create fixed-length epochs, automatically estimate an amplitude rejection threshold,
   then fit ICA and exclude eye-movement (EOG) components.


In [ ]:
# ── Channel type assignment ─────────────────────────────────────────────────
# MNE needs to know which channels are EEG vs. EOG vs. ECG so that it can
# apply artifact detection and montage correctly.
channel_types = {
    'LE':  'eog',   # Left eye electrode
    'RE':  'eog',   # Right eye electrode
    'ECG': 'ecg',   # Cardiac channel
    'Ne':  'misc',  # Non-EEG / miscellaneous
    'Ma':  'misc',
    'Ext': 'misc',
}
raw.set_channel_types(channel_types)

# Apply the standard 10-05 electrode position system
montage = mne.channels.make_standard_montage('standard_1005')
raw.set_montage(montage)

In [ ]:
# ── Band-pass filtering (for downstream epoching & ERPs) ───────────────────
# 0.1 Hz high-pass removes slow drifts; 30 Hz low-pass removes high-frequency
# noise and line interference above the gamma band of interest.
low_cut = 0.1
hi_cut  = 30.0

raw_filt = raw.copy().filter(low_cut, hi_cut)

# Visualise the power spectral density to confirm the filter worked
raw_filt.plot_psd(fmax=40, show=False)
plt.show()

In [ ]:
# ── ICA preparation ─────────────────────────────────────────────────────────
# ICA works best when low-frequency power is removed more aggressively (≥1 Hz).
# We create a separate filtered copy solely for ICA fitting.
ica_low_cut = 1.0
raw_ica     = raw.copy().filter(ica_low_cut, hi_cut)

# Segment the ICA copy into 1-second fixed-length epochs.
# These are used only for fitting ICA — not for downstream analysis.
tstep      = 1.0
events_ica = mne.make_fixed_length_events(raw_ica, duration=tstep)
epochs_ica = mne.Epochs(
    raw_ica, events_ica,
    tmin=0.0, tmax=tstep,
    baseline=None, preload=True
)

In [ ]:
from autoreject import get_rejection_threshold

# Automatically compute a peak-to-peak amplitude rejection threshold.
# Epochs exceeding this threshold are excluded from ICA fitting,
# preventing large artefacts from corrupting the decomposition.
reject = get_rejection_threshold(epochs_ica)
print('Rejection threshold:', reject)

In [ ]:
# ── Fit ICA ─────────────────────────────────────────────────────────────────
random_state    = 42    # Fix seed for reproducibility
ica_n_components = 0.99  # Retain components explaining 99 % of variance

ica = mne.preprocessing.ICA(n_components=ica_n_components, random_state=random_state)
ica.fit(epochs_ica, reject=reject, tstep=tstep)

In [ ]:
# ── Identify and exclude EOG artefact components ───────────────────────────
# Uses frontal channels (Fp1, F8) as EOG proxies via z-score correlation.
# Components with |z| > 1.96 (≈ 95 % CI) are flagged as eye-movement artefacts.
ica_z_thresh = 1.96
eog_indices, eog_scores = ica.find_bads_eog(
    raw_ica,
    ch_name=['Fp1', 'F8'],
    threshold=ica_z_thresh
)
ica.exclude = eog_indices
print(f'Excluded ICA components (EOG): {eog_indices}')

# Visualise the z-scores — flagged components shown in red
ica.plot_scores(eog_scores)
plt.show()

# Inspect the spatial topographies of all ICA components
ica.plot_components()
plt.show()

## 4. Epoching & Visualisation

Create stimulus-locked epochs around each pain-level event, apply ICA artefact
correction, re-reference to mastoid electrodes, and visualise the resulting ERPs.


In [ ]:
# ── Epoch the filtered (non-ICA) data around stimulus events ───────────────
tmin     = -0.200  # 200 ms pre-stimulus baseline
tmax     =  1.000  # 1 000 ms post-stimulus
baseline = (None, 0)  # Baseline-correct using the pre-stimulus window

epochs = mne.Epochs(
    raw_filt,
    events, event_mapping,
    tmin, tmax,
    baseline=baseline,
    preload=True
)
print(epochs)

In [ ]:
# Quick sanity check: visualise the grand-average ERP before artefact removal
epochs.average().plot(spatial_colors=True, show=False)
plt.title('Grand-average ERP — before ICA correction')
plt.show()

# Scalp topography at 100 ms intervals across the epoch window
times = np.arange(0, tmax, 0.1)
epochs.average().plot_topomap(times=times, average=0.050)
plt.show()

In [ ]:
# ── Apply ICA to the stimulus-locked epochs ─────────────────────────────────
# Removes the previously identified EOG components from the data.
epochs_postica = ica.apply(epochs.copy())

# Visualise grand-average ERP after artefact correction
epochs_postica.average().plot(spatial_colors=True, show=False)
plt.title('Grand-average ERP — after ICA correction')
plt.show()

times = np.arange(0, tmax, 0.1)
epochs_postica.average().plot_topomap(times=times, average=0.050)
plt.show()

In [ ]:
# ── Re-reference to linked mastoids ─────────────────────────────────────────
# TP9 and TP10 correspond to the left and right mastoid electrodes.
# Mastoid referencing is standard in ERP and pain research.
epochs_mastoidref = epochs_postica.set_eeg_reference(ref_channels=['TP9', 'TP10'])

times = np.arange(0, tmax, 0.1)
epochs_mastoidref.average().plot_topomap(times=times, average=0.050)
plt.title('Grand-average topography — mastoid reference')
plt.show()

In [ ]:
# ── Per-condition ERP plots ──────────────────────────────────────────────────
# Create one Evoked object per pain-level condition for comparison
conditions = [
    'Comment/10', 'Comment/20', 'Comment/30',
    'Comment/40', 'Comment/50', 'Comment/60',
]
evokeds = {c: epochs_mastoidref[c].average() for c in conditions}

# Assign condition name as comment so it appears in plot titles
for condition, evoked in evokeds.items():
    evoked.comment = condition

# Joint plot shows butterfly + topomaps at key latencies
key_times = [0.150, 0.250, 0.400, 0.600, 0.800]
for condition, evoked in evokeds.items():
    evoked.plot_joint(times=key_times, title=condition)
    plt.show()

## 5. DWT Feature Extraction

For each epoch and each EEG channel, apply a 5-level Daubechies-4 Discrete
Wavelet Transform and extract 8 statistical features from the resulting
approximation (cA) and detail (cD) coefficients:

| Feature | Description |
|---|---|
| `cD_Energy` | Mean energy across detail sub-bands |
| `cA_Energy` | Energy of the approximation coefficients |
| `D_Entropy` | Mean wavelet entropy across detail sub-bands |
| `A_Entropy` | Wavelet entropy of the approximation coefficients |
| `D_mean` | Mean amplitude across detail sub-bands |
| `A_mean` | Mean amplitude of the approximation coefficients |
| `D_std` | Mean std deviation across detail sub-bands |
| `A_std` | Std deviation of the approximation coefficients |

This yields **8 features × N channels** per epoch, saved to a per-participant CSV.


In [ ]:
from pywt import wavedec

# EEG channels used for feature extraction.
# These 8 channels span frontal, central, parietal, and occipital regions.
CHANNELS = ['Fz', 'C3', 'Cz', 'C4', 'Pz', 'PO7', 'Oz', 'PO8']


def wavelet_avg_features(data, type_wav='db4'):
    """
    Compute 8 DWT-based features from a single EEG channel signal.

    Parameters
    ----------
    data : array-like, shape (n_samples,)
        Single-channel EEG time series.
    type_wav : str
        Wavelet family to use. Default is 'db4' (Daubechies-4).

    Returns
    -------
    list of float
        [cD_Energy, cA_Energy, D_Entropy, A_Entropy,
         D_mean, A_mean, D_std, A_std]
    """
    # Decompose signal into 5 detail sub-bands + 1 approximation sub-band
    coeffs = wavedec(data, type_wav, level=5)
    # coeffs[0]   → approximation (cA5)
    # coeffs[1..5] → detail sub-bands (cD5 … cD1, fine → coarse)

    # Energy: sum of squared coefficients (Parseval's theorem analog)
    cD_Energy = np.mean([np.sum(np.square(coeffs[i])) for i in range(1, 6)])
    cA_Energy = np.sum(np.square(coeffs[0]))

    # Wavelet entropy: captures signal complexity / information content
    D_Entropy = np.mean([
        np.sum(np.square(coeffs[i]) * np.log(np.square(coeffs[i]) + 1e-10))
        for i in range(1, 6)
    ])
    A_Entropy = np.sum(np.square(coeffs[0]) * np.log(np.square(coeffs[0]) + 1e-10))

    # Statistical moments
    D_mean = np.mean([np.mean(coeffs[i]) for i in range(1, 6)])
    A_mean = np.mean(coeffs[0])
    D_std  = np.mean([np.std(coeffs[i]) for i in range(1, 6)])
    A_std  = np.std(coeffs[0])

    return [cD_Energy, cA_Energy, D_Entropy, A_Entropy, D_mean, A_mean, D_std, A_std]

In [ ]:
# ── Step 1: Extract pain-level labels from annotations ──────────────────────
# Each 'Comment/XX' annotation encodes a pain stimulus intensity (10–60).
# We divide by 10 so labels become integers 1–6.
print('Extracting labels from annotations...')

labels = []
for ann in raw.annotations:
    if 'Comment/' in ann['description']:
        try:
            raw_level = ann['description'].split('/')[-1]
            # Remove any non-digit characters (e.g. 'ß') before converting
            pain_level = int(raw_level.replace('ß', '')) // 10
            labels.append(pain_level)
        except ValueError:
            print(f'Warning: skipping invalid annotation — {ann["description"]}')

print(f'Extracted {len(labels)} labels.')

In [ ]:
# ── Step 2: Extract wavelet features for every epoch ───────────────────────
features = []

for epoch in epochs.iter_evoked():
    epoch_features = []
    for ch_idx in range(len(CHANNELS)):
        # wavelet_avg_features returns 8 values per channel
        ch_features = wavelet_avg_features(epoch.data[ch_idx, :])
        epoch_features.extend(ch_features)
    features.append(epoch_features)

# ── Step 3: Align features and labels (lengths may differ slightly) ─────────
min_length = min(len(features), len(labels))
features   = features[:min_length]
labels     = labels[:min_length]

print(f'Aligned features and labels: {min_length} rows.')

In [ ]:
# ── Step 4: Save features + labels to a per-participant CSV ─────────────────
output_dir = '/content/drive/My Drive/discrete_wavelet/8channels/'

# Build column names: one set of 8 feature names per channel
columns = []
feature_names = ['cD_Energy', 'cA_Energy', 'D_Entropy', 'A_Entropy',
                 'D_mean',    'A_mean',    'D_std',     'A_std']
for ch in CHANNELS:
    columns.extend([f'{ch}_{feat}' for feat in feature_names])

df_participant = pd.DataFrame(features, columns=columns)
df_participant['Label'] = labels

csv_path = os.path.join(output_dir, f'wavelet_{p_id}.csv')
df_participant.to_csv(csv_path, index=False)

print(f'Saved {df_participant.shape[0]} epochs × {df_participant.shape[1]} columns → {csv_path}')

## 6. Merging Per-Participant CSV Files

After running the extraction above for every participant, merge all individual
CSVs into a single dataset file ready for model training.


In [ ]:
# ── Merge all per-participant CSVs ──────────────────────────────────────────
folder_path = '/content/drive/My Drive/discrete_wavelet/8channels'
output_file = '/content/drive/My Drive/discrete_wavelet/merged/merged_data_8channels.csv'

csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
print(f'Found {len(csv_files)} participant file(s): {csv_files}')

df_list = [pd.read_csv(os.path.join(folder_path, f)) for f in csv_files]
merged_df = pd.concat(df_list, ignore_index=True)

merged_df.to_csv(output_file, index=False)
print(f'Merged dataset saved → {output_file}')
print(f'Shape: {merged_df.shape}')

## 7. Dataset Preparation

Load the merged dataset, verify integrity, and apply optional moving-average
smoothing within each pain-level condition before model training.


In [ ]:
# ── Load merged dataset ─────────────────────────────────────────────────────
csv_path = '/content/drive/My Drive/discrete_wavelet/merged/merged_data_8channels.csv'
df = pd.read_csv(csv_path)

print(df.head())
print(f'\nDataset shape: {df.shape}')
print(f'Label distribution:\n{df["Label"].value_counts().sort_index()}')

# Check for missing values — none expected, but good practice
missing_X = df.drop(columns=['Label']).isna().sum().sum()
missing_Y = df['Label'].isna().sum()
print(f'\nMissing values — Features: {missing_X} | Labels: {missing_Y}')

In [ ]:
# ── Feature correlation heatmap ─────────────────────────────────────────────
# Helps identify highly correlated features that may be redundant.
# Large figures are needed due to the number of feature columns.
plt.figure(figsize=(50, 50))
cor_matrix = df.corr()
sns.heatmap(cor_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# ── Optional: moving-average smoothing per pain-level class ─────────────────
# Smoothing reduces sample-to-sample noise within each condition, which can
# improve the signal-to-noise ratio seen by the classifier.

def moving_average(x, w=10):
    """Apply a simple uniform moving average with window size w."""
    return np.convolve(x, np.ones(w), 'valid') / w


df_true = df.copy()
df_true['predefinedlabel'] = df_true['Label']

feature_columns = CHANNELS  # The 8 channels defined in Section 5
feature_suffixes = ['cD_Energy', 'cA_Energy', 'D_Entropy', 'A_Entropy',
                    'D_mean',    'A_mean',    'D_std',     'A_std']

df_ma_rows = []

for k in df_true['predefinedlabel'].unique():
    df_k = df_true[df_true['predefinedlabel'] == k]
    features_smoothed = {}

    for col in feature_columns:
        for suffix in feature_suffixes:
            feat_col = f'{col}_{suffix}'
            features_smoothed[feat_col] = moving_average(df_k[feat_col].values)

    n = len(next(iter(features_smoothed.values())))
    label_arr     = np.full(n, k)
    timepoint_arr = np.arange(n)

    row_df = pd.DataFrame(features_smoothed)
    row_df['timepoint'] = timepoint_arr
    row_df['Label']     = label_arr
    df_ma_rows.append(row_df)

df_ma = pd.concat(df_ma_rows, ignore_index=True)

print(f'Smoothed dataset shape: {df_ma.shape}')
print(df_ma.head())

## 8. LightGBM Model Training & Evaluation

Train a LightGBM multi-class classifier on the smoothed feature dataset.
Two complementary evaluations are used:
- **Hold-out test set** (80/20 split): gives a single accuracy estimate.
- **Stratified 3-fold cross-validation**: gives a more robust, variance-aware estimate.


In [ ]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
from sklearn.preprocessing import StandardScaler


In [ ]:
# ── Prepare features and labels ─────────────────────────────────────────────
X = df_ma.drop(columns=['Label'])
Y = df_ma['Label']

# Ensure all feature columns are numeric
X = X.select_dtypes(include=['number']).astype('float32')

print(f'Feature matrix: {X.shape}')
print(f'Label distribution:\n{Y.value_counts().sort_index()}')

In [ ]:
# ── Train / test split ──────────────────────────────────────────────────────
# stratify=Y ensures each pain-level class is proportionally represented
# in both the training and test partitions.
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# Feature scaling — XGBoost is tree-based and does not strictly require
# scaling, but it can help convergence when combined with regularisation.
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

In [ ]:
# ── Define and train the LightGBM classifier ─────────────────────────────────
# Key hyperparameters:
#   objective      : 'multiclass' for discrete class outputs
#   num_class      : number of unique pain-level labels
#   learning_rate  : step size shrinkage to prevent overfitting
#   n_estimators   : number of boosting rounds
model = LGBMClassifier(
    objective='multiclass',
    num_class=int(Y.max()) + 1,  # Automatically inferred from data
    learning_rate=0.1,
    n_estimators=100,
    random_state=42,
)

model.fit(X_train, y_train)
print('Training complete.')


In [ ]:
# ── Evaluate on the hold-out test set ───────────────────────────────────────
y_pred = model.predict(X_test)

accuracy_test = accuracy_score(y_test, y_pred)
f1_test       = f1_score(y_test, y_pred, average='weighted')

print(f'Test Accuracy : {accuracy_test * 100:.2f}%')
print(f'Test F1 Score : {f1_test:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# ── Confusion matrix ────────────────────────────────────────────────────────
conf_matrix = confusion_matrix(y_test, y_pred)
class_labels = sorted(Y.unique())

plt.figure(figsize=(8, 6))
sns.heatmap(
    conf_matrix, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_labels, yticklabels=class_labels
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix — Hold-out Test Set')
plt.tight_layout()
plt.show()

In [ ]:
# ── Stratified 3-fold cross-validation ─────────────────────────────────────
# Cross-validation gives a more reliable accuracy estimate by training and
# evaluating on all folds of the data, reducing dependence on a single split.
skf        = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_results = cross_val_score(model, X, Y, cv=skf, scoring='accuracy')

print(f'Cross-validation Accuracy: {cv_results.mean() * 100:.2f}% '
      f'(± {cv_results.std() * 100:.2f}%)')
print(f'Per-fold scores: {[f"{s*100:.1f}%" for s in cv_results]}')


In [ ]:
# ── Feature importance ──────────────────────────────────────────────────────
# LightGBM's 'split' importance counts the number of times a feature is used
# to split a node. High-importance features are the most discriminative for
# pain classification.
import pandas as pd
import matplotlib.pyplot as plt

feat_imp = pd.Series(
    model.feature_importances_,
    index=X.columns if hasattr(X, 'columns') else range(len(model.feature_importances_))
).nlargest(20)

plt.figure(figsize=(12, 6))
feat_imp.sort_values().plot(kind='barh')
plt.title('Top-20 Feature Importances (LightGBM split count)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 9. Model Saving

Export the trained LightGBM model to Google Drive so it can be loaded later
for inference or fine-tuning without retraining.


In [ ]:
# ── Save model weights ──────────────────────────────────────────────────────
# LightGBM models are saved in text format using the built-in save_model method.
model_path = '/content/drive/My Drive/lightgbm_model_8channels.txt'
model.booster_.save_model(model_path)
print(f'Model saved → {model_path}')
